# Betweenness Centrality Variants on the Xujianian Similarity Network

Supplementary analysis for "Peripheral Elites: Status-Centrality Decoupling in Social Networks" (Xujianian cemetery, Siwa culture, twelfth to eleventh centuries BCE).

Standard betweenness centrality is known to be unstable on dense weighted networks: small fluctuations in edge weights can dramatically reroute shortest paths, so the betweenness score of any given node can depend on narrow margins in a few individual ties (Segarra & Ribeiro, 2015). For a similarity network with density 0.606 this instability is a concrete concern, and it is compounded by the fact that shortest-path centralities on similarity networks require a choice between several possible weighting conventions that can produce different answers on the same network.

This notebook implements two stability-oriented betweenness variants and compares them against three baseline specifications of standard betweenness on the Xujianian similarity network.

**Baselines**
- **B1.** Unweighted betweenness
- **B2.** Similarity-as-weight betweenness (matches the main text Section 4.2 primary specification)
- **B3.** Distance-as-weight betweenness (distance = 1 − similarity)

**Variants**
- **V1.** Stabler betweenness (Segarra & Ribeiro, 2015)
- **V2.** Gromov centrality (Babul & Reggiani, 2022)

For each measure we compute the rare-goods to non-rare centrality ratio and run a 10,000-iteration permutation test matching the node-level inference protocol used in the main text.

**Convention note.** NetworkX's `betweenness_centrality` treats edge weight as distance (cost). Passing similarity directly therefore produces shortest paths that preferentially traverse dissimilar pairs, which is not the intended topology for a shortest-path centrality on a similarity network. The technically consistent convention is to use distance = 1 − similarity; the variants in this notebook use that convention. All three baselines are reported side by side to document the sensitivity of standard betweenness to the convention chosen.

## 1. Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## 2. Data and rare-goods classification

Load the 104-burial Xujianian dataset and apply the same rare-goods classification used in the main text: any artifact type present in fewer than 10% of burials is coded as rare, and a burial is coded as a rare-goods holder if it contains at least one rare artifact.

In [ ]:
df = pd.read_csv('../data/xujianian_english_final.csv')

ARTIFACT_COLS = [
    'pottery_count', 'bronze_count', 'bone_count', 'jade_count',
    'agate_count', 'turquoise_count', 'stone_count',
    'shell_ornament_count', 'cowrie_count', 'spindle_whorl'
]

df_binary = (df[ARTIFACT_COLS] > 0).astype(int)
df_binary.index = df['burial_id']

RARE_THRESHOLD = 0.10
presence_rates = (df[ARTIFACT_COLS] > 0).mean()
rare_artifacts = presence_rates[presence_rates < RARE_THRESHOLD].index.tolist()
df['has_rare_goods'] = (df[rare_artifacts].sum(axis=1) > 0).astype(int)

print(f'Rare artifact types (<10% presence): {rare_artifacts}')
print(f'Burials with rare goods: {df["has_rare_goods"].sum()} / {len(df)}')

## 3. Network construction

Build the Jaccard similarity network at threshold 0.30, identical to the main text Section 3.2. Each edge carries both the raw similarity (`weight`) and the corresponding distance (`distance = 1 − similarity`), so we can compute centralities under either convention.

In [ ]:
def jaccard_similarity(a, b):
    inter = np.sum((a == 1) & (b == 1))
    union = np.sum((a == 1) | (b == 1))
    return inter / union if union > 0 else 0

n = len(df_binary)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        s = jaccard_similarity(df_binary.iloc[i].values, df_binary.iloc[j].values)
        sim_matrix[i, j] = s
        sim_matrix[j, i] = s

THRESHOLD = 0.3
G = nx.Graph()
G.add_nodes_from(df_binary.index)
for i, n1 in enumerate(df_binary.index):
    for j, n2 in enumerate(df_binary.index):
        if i < j and sim_matrix[i, j] >= THRESHOLD:
            G.add_edge(n1, n2,
                       weight=sim_matrix[i, j],
                       distance=1.0 - sim_matrix[i, j] + 1e-9)

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, density = {nx.density(G):.4f}')

rare_map = dict(zip(df['burial_id'], df['has_rare_goods']))
rare_nodes = [n for n in G.nodes() if rare_map[n] == 1]
nonrare_nodes = [n for n in G.nodes() if rare_map[n] == 0]
print(f'Rare-goods holders: {len(rare_nodes)}, non-holders: {len(nonrare_nodes)}')

## 4. Helper functions: group ratio and permutation test

The permutation test used throughout this analysis matches the node-level protocol used in the main text: 10,000 random assignments of the rare-goods label, two-sided test on the absolute deviation of the observed ratio from 1.

In [ ]:
def group_ratio(centrality, rare, nonrare):
    rm = np.mean([centrality[n] for n in rare])
    nm = np.mean([centrality[n] for n in nonrare])
    return rm / nm if nm > 0 else float('inf'), rm, nm


def permutation_test(centrality, rare, nonrare, n_perms=10000, seed=42):
    rng = np.random.default_rng(seed)
    nodes = list(centrality.keys())
    n_rare = len(rare)
    obs_ratio, _, _ = group_ratio(centrality, rare, nonrare)
    arr = np.array([centrality[n] for n in nodes])
    n_total = len(nodes)
    null_ratios = np.empty(n_perms)
    for k in range(n_perms):
        idx = rng.permutation(n_total)
        rm = arr[idx[:n_rare]].mean()
        nm = arr[idx[n_rare:]].mean()
        null_ratios[k] = rm / nm if nm > 0 else np.nan
    null_ratios = null_ratios[~np.isnan(null_ratios)]
    p = (np.sum(np.abs(null_ratios - 1) >= abs(obs_ratio - 1)) + 1) / (len(null_ratios) + 1)
    return obs_ratio, p


def report(label, centrality, rare, nonrare):
    obs, p = permutation_test(centrality, rare, nonrare, n_perms=10000)
    rm = np.mean([centrality[n] for n in rare])
    nm = np.mean([centrality[n] for n in nonrare])
    print(f'{label}:')
    print(f'  Mean (rare):     {rm:.6f}')
    print(f'  Mean (non-rare): {nm:.6f}')
    print(f'  Ratio:           {obs:.4f}')
    print(f'  Permutation p:   {p:.4f}')
    return {'rare_mean': float(rm), 'nonrare_mean': float(nm),
            'ratio': float(obs), 'p_value': float(p)}

## 5. Baseline betweenness under three conventions

### B1. Unweighted betweenness

Treats all edges as equivalent. This corresponds to the unweighted robustness check reported in the main text (Section 4.6, Table 8).

In [ ]:
bc_unw = nx.betweenness_centrality(G, normalized=True)
res_b1 = report('B1. Unweighted betweenness', bc_unw, rare_nodes, nonrare_nodes)

### B2. Similarity-as-weight betweenness

Passes the raw Jaccard similarity to NetworkX as the edge weight. This matches the primary specification used in the main text Section 4.2. NetworkX treats the weight as distance, so under this convention shortest paths preferentially traverse dissimilar pairs.

In [ ]:
bc_simw = nx.betweenness_centrality(G, weight='weight', normalized=True)
res_b2 = report('B2. Similarity-as-weight betweenness', bc_simw, rare_nodes, nonrare_nodes)

### B3. Distance-as-weight betweenness

Uses distance = 1 − similarity, the technically consistent convention for shortest-path centrality on a similarity network: shortest paths preferentially traverse the most similar pairs.

In [ ]:
bc_distw = nx.betweenness_centrality(G, weight='distance', normalized=True)
res_b3 = report('B3. Distance-as-weight betweenness', bc_distw, rare_nodes, nonrare_nodes)

## 6. Stabler betweenness (Segarra & Ribeiro, 2015)

Segarra and Ribeiro (2015) introduce stabler betweenness as a remedy for the instability of standard betweenness on dense weighted networks. In such networks, small changes in edge weights can dramatically reroute shortest paths, so a node's betweenness score can depend on narrow margins in one or two individual ties. The solution is to average standard betweenness across multiple realisations of a small random perturbation of the edge weights. The resulting estimate approximates the expected betweenness a node would receive under small fluctuations in tie strength and is substantially less sensitive to tie-level noise.

We implement the method using 100 perturbations at Gaussian standard deviation σ = 0.10 × original similarity, with perturbed values clipped to [0.001, 0.999] and converted to distance for input to `betweenness_centrality`.

In [ ]:
def stabler_betweenness(G, n_perturbations=100, sigma=0.10, seed=42):
    """Stabler betweenness via repeated edge-weight perturbation
    (Segarra & Ribeiro, 2015)."""
    rng = np.random.default_rng(seed)
    nodes = list(G.nodes())
    accum = {n: 0.0 for n in nodes}
    for k in range(n_perturbations):
        Gp = G.copy()
        for u, v, d in Gp.edges(data=True):
            sim_orig = d['weight']
            sim_pert = max(0.001, min(0.999, sim_orig + rng.normal(0, sigma * sim_orig)))
            d['weight'] = sim_pert
            d['distance'] = 1.0 - sim_pert + 1e-9
        bc = nx.betweenness_centrality(Gp, weight='distance', normalized=True)
        for n in nodes:
            accum[n] += bc[n]
    return {n: accum[n] / n_perturbations for n in nodes}


bc_stab = stabler_betweenness(G, n_perturbations=100, sigma=0.10, seed=42)
res_stab = report('V1. Stabler betweenness (n=100, sigma=0.10)', bc_stab, rare_nodes, nonrare_nodes)

### Parameter sensitivity

Stabler betweenness has two parameters: the number of perturbed instances and the standard deviation of the perturbation as a fraction of the original similarity. To check that the result is not an artifact of particular parameter choices, we re-run the analysis across a 3×3 grid.

In [ ]:
sensitivity = {}
for n_p in [50, 100, 200]:
    for sg in [0.05, 0.10, 0.15]:
        bc_s = stabler_betweenness(G, n_perturbations=n_p, sigma=sg, seed=42)
        r, p = permutation_test(bc_s, rare_nodes, nonrare_nodes, n_perms=2000, seed=43)
        sensitivity[(n_p, sg)] = (r, p)
        print(f'  n={n_p}, sigma={sg}: ratio={r:.4f}, p={p:.4f}')

## 7. Gromov centrality (Babul & Reggiani, 2022)

Babul and Reggiani (2022) define Gromov centrality through the four-point Gromov product from metric geometry:

$$(x \mid y)_v = \tfrac{1}{2}\big( d(x, v) + d(y, v) - d(x, y) \big)$$

The Gromov product is the triangle inequality excess for the triple $(x, y, v)$, and it measures how far $v$ lies from the geodesic between $x$ and $y$. Averaging $(x \mid y)_v$ over all pairs $(x, y)$ distinct from $v$ yields a per-node score: nodes that lie on many geodesics have a small average Gromov product (low triangle inequality excess), nodes off to the side of the metric structure have a large average.

Gromov centrality inverts the sign of this score so that high values correspond to central positions, matching the convention used for other centrality measures:

$$\mathrm{GC}(v) = \max_{v'} \overline{(x \mid y)_{v'}} - \overline{(x \mid y)_v}$$

Unlike standard betweenness, which is built from shortest-path counts, Gromov centrality is defined from the full metric structure of the graph, so it remains well-defined even when shortest paths themselves are unstable.

We compute Gromov centrality exactly by enumerating all triples, which is tractable for the Xujianian network (104 nodes, ~175,000 triples).

In [ ]:
def gromov_centrality(G, weight='distance'):
    """Gromov centrality (Babul & Reggiani, 2022, Phys. Rev. E 106, 034312)."""
    nodes = list(G.nodes())
    n = len(nodes)
    node_idx = {nd: i for i, nd in enumerate(nodes)}
    dist_dict = dict(nx.all_pairs_dijkstra_path_length(G, weight=weight))
    D = np.full((n, n), np.inf)
    for u in dist_dict:
        for v, d in dist_dict[u].items():
            D[node_idx[u], node_idx[v]] = d
    finite_D = D[np.isfinite(D)]
    diameter = finite_D.max() if len(finite_D) > 0 else 1.0
    D[np.isinf(D)] = diameter

    avg_gp = np.zeros(n)
    for v_i in range(n):
        gp_sum = 0.0
        cnt = 0
        for x_i in range(n):
            if x_i == v_i:
                continue
            for y_i in range(x_i + 1, n):
                if y_i == v_i:
                    continue
                gp = 0.5 * (D[x_i, v_i] + D[y_i, v_i] - D[x_i, y_i])
                gp_sum += gp
                cnt += 1
        avg_gp[v_i] = gp_sum / cnt if cnt > 0 else 0.0

    max_gp = avg_gp.max()
    return {nodes[i]: float(max_gp - avg_gp[i]) for i in range(n)}


gc_gromov = gromov_centrality(G, weight='distance')
res_gromov = report('V2. Gromov centrality', gc_gromov, rare_nodes, nonrare_nodes)

## 8. Summary table

In [ ]:
table = pd.DataFrame([
    {'Measure': 'B1. Unweighted betweenness',
     'Mean (rare)': f"{res_b1['rare_mean']:.6f}",
     'Mean (non-rare)': f"{res_b1['nonrare_mean']:.6f}",
     'Ratio': f"{res_b1['ratio']:.3f}",
     'Permutation p': f"{res_b1['p_value']:.4f}"},
    {'Measure': 'B2. Similarity-as-weight betweenness',
     'Mean (rare)': f"{res_b2['rare_mean']:.6f}",
     'Mean (non-rare)': f"{res_b2['nonrare_mean']:.6f}",
     'Ratio': f"{res_b2['ratio']:.3f}",
     'Permutation p': f"{res_b2['p_value']:.4f}"},
    {'Measure': 'B3. Distance-as-weight betweenness',
     'Mean (rare)': f"{res_b3['rare_mean']:.6f}",
     'Mean (non-rare)': f"{res_b3['nonrare_mean']:.6f}",
     'Ratio': f"{res_b3['ratio']:.3f}",
     'Permutation p': f"{res_b3['p_value']:.4f}"},
    {'Measure': 'V1. Stabler betweenness',
     'Mean (rare)': f"{res_stab['rare_mean']:.6f}",
     'Mean (non-rare)': f"{res_stab['nonrare_mean']:.6f}",
     'Ratio': f"{res_stab['ratio']:.3f}",
     'Permutation p': f"{res_stab['p_value']:.4f}"},
    {'Measure': 'V2. Gromov centrality',
     'Mean (rare)': f"{res_gromov['rare_mean']:.6f}",
     'Mean (non-rare)': f"{res_gromov['nonrare_mean']:.6f}",
     'Ratio': f"{res_gromov['ratio']:.3f}",
     'Permutation p': f"{res_gromov['p_value']:.4f}"},
])
table

## 9. Figures

### Figure S1. Rare vs non-rare centrality distributions across the five measures

In [ ]:
variants_data = [
    ('B1. Unweighted', bc_unw, res_b1),
    ('B2. Similarity-as-weight', bc_simw, res_b2),
    ('B3. Distance-as-weight', bc_distw, res_b3),
    ('V1. Stabler', bc_stab, res_stab),
    ('V2. Gromov centrality', gc_gromov, res_gromov),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, (title, cent, res) in zip(axes, variants_data):
    rare_vals = [cent[n] for n in rare_nodes]
    nonrare_vals = [cent[n] for n in nonrare_nodes]
    bp = ax.boxplot([nonrare_vals, rare_vals],
                    tick_labels=['Non-rare\n(n=86)', 'Rare-goods\n(n=18)'],
                    patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], ['#6baed6', '#e6550d']):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    sig = '***' if res['p_value'] < 0.001 else ('**' if res['p_value'] < 0.01 else ('*' if res['p_value'] < 0.05 else 'ns'))
    ax.set_title(f"{title}\nratio = {res['ratio']:.2f}, p = {res['p_value']:.4f} {sig}", fontsize=10)
    ax.set_ylabel('Centrality')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('betweenness_variants_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Figure S2. Stabler betweenness parameter sensitivity

In [ ]:
sens_df = pd.DataFrame(
    [[sensitivity[(n_p, sg)][0] for sg in [0.05, 0.10, 0.15]] for n_p in [50, 100, 200]],
    index=['n=50', 'n=100', 'n=200'],
    columns=['sigma=0.05', 'sigma=0.10', 'sigma=0.15']
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(sens_df, annot=True, fmt='.3f', cmap='RdBu_r', center=1.0,
            cbar_kws={'label': 'Rare/non-rare ratio'}, ax=ax,
            vmin=0.1, vmax=0.4)
ax.set_title('Stabler Betweenness: Parameter Sensitivity\n(rare/non-rare ratio)', fontsize=11)
plt.tight_layout()
plt.savefig('stabler_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Save results

In [ ]:
results = {
    'baseline_unweighted': {**res_b1, 'name': 'Unweighted betweenness'},
    'baseline_sim_weight': {**res_b2, 'name': 'Similarity-as-weight betweenness'},
    'baseline_dist_weight': {**res_b3, 'name': 'Distance-as-weight betweenness'},
    'stabler': {**res_stab, 'name': 'Stabler betweenness',
                'reference': 'Segarra & Ribeiro (2015)',
                'parameters': {'n_perturbations': 100, 'sigma': 0.10}},
    'gromov': {**res_gromov, 'name': 'Gromov centrality',
               'reference': 'Babul & Reggiani (2022)'},
    'sensitivity': {f'n={k[0]},sigma={k[1]}': {'ratio': float(v[0]), 'p_value': float(v[1])}
                    for k, v in sensitivity.items()}
}

with open('betweenness_variants_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to betweenness_variants_results.json')

## 11. Interpretation

The five measures fall into three groups.

**Baseline betweenness (B1–B3).** All three conventions produce rare/non-rare ratios close to 1 with p-values that fail to reach conventional significance. The direction of the ratio changes between conventions (unweighted 0.66, similarity-as-weight 1.48, distance-as-weight 0.96), which is the specific kind of operationalisation sensitivity that motivates stability-oriented alternatives (Segarra & Ribeiro, 2015).

**Stabler betweenness (V1).** Under 100 Gaussian edge-weight perturbations at σ = 0.10, rare-goods holders have a mean stabler betweenness roughly one-fifth that of non-holders (ratio = 0.22, p < .001). The effect is stable across the sensitivity grid: ratios between 0.22 and 0.27, p ≤ .002 at every parameter combination.

**Gromov centrality (V2).** Rare-goods holders have a mean Gromov centrality about 70% that of non-holders (ratio = 0.71, p < .001), meaning their positions are on average further from the geodesic structure of the network.

The two stability-oriented variants operationalise metric centrality in fundamentally different ways—perturbation averaging of shortest-path counts for V1, four-point Gromov product for V2—yet they converge on the same finding: rare-goods holders are peripheral rather than bridging when betweenness is measured in a way that is robust to the instability of standard shortest-path counts on a dense weighted network. This peripheral pattern is consistent with the disembedding framework developed in Sections 2.3 and 5.2 of the main text, under which actors whose distinction derives from access to sources outside the observable network are not expected to occupy within-network bridging positions: any bridges such actors occupy connect to destinations that the similarity network, by construction, cannot see.